# Spark's lazy engine: transformations vs actions

_Apache Spark (big-data processing)_

**Transformations only build a plan; an action triggers the whole computation — and Spark optimizes the chain before running it.**

Spark splits the world into two kinds of operation, and the split is the whole trick.

       
         
- Transformations are lazy. select, filter, withColumn,
         groupBy, join &mdash; each one returns a new DataFrame instantly and computes
         nothing. All it does is add a step to a recipe. That recipe is a DAG (Directed Acyclic Graph):
         a graph of steps where the arrows only ever point forward (no step depends on itself, directly or
         indirectly), so there is a clear order to run them in.
         
- Actions are eager. show, count, collect,
         write, take &mdash; calling one says "I actually want a result now." Only then
         does Spark look at the whole DAG you built, plan it, and run it across the cluster.
       

---

This notebook is a practice scaffold for the **Spark's lazy engine: transformations vs actions** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PySpark

In [ ]:
# Colab: !pip install pyspark   (run once in a setup cell)
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")            # local mode: one machine, all cores
         .appName("lazy-demo")
         .getOrCreate())
# Default is 200 shuffle partitions; keep it so the Exchange is easy to see.
spark.conf.set("spark.sql.shuffle.partitions", 200)

# --- Build a small DataFrame and spread it across 8 partitions -----------
rows = [(i, ["west", "east", "north", "south", "central"][i % 5], float(i % 100) - 20)
        for i in range(120_000)]
df = spark.createDataFrame(rows, ["id", "region", "amount"]).repartition(8)
print("input partitions:", df.rdd.getNumPartitions())   # -> 8

# --- TRANSFORMATIONS: all LAZY. Nothing runs here. ----------------------
t0 = time.time()
plan = (df
        .filter(F.col("amount") > 0)        # NARROW: per-partition, no shuffle
        .withColumn("amt2", F.col("amount") * 2)  # NARROW
        .groupBy("region")                  # WIDE: forces a SHUFFLE (Exchange)
        .agg(F.sum("amount").alias("total"),
             F.count("*").alias("n")))
print("built the plan in %.4fs (nothing computed yet)" % (time.time() - t0))
# -> a few milliseconds: Spark only recorded the recipe (the DAG).

# --- Read the plan: the Exchange is the shuffle = the stage boundary -----
plan.explain()
# Look for a line like:
#   *(2) HashAggregate(keys=[region], ...)
#   +- Exchange hashpartitioning(region, 200)   <-- SHUFFLE: end of Stage 1
#      +- *(1) HashAggregate(...)                <-- partial agg, Stage 1
#         +- *(1) Filter (amount > 0)            <-- NARROW, Stage 1
#            +- Scan ...                          (read)

# --- ACTION: NOW the whole DAG runs as a job -> 2 stages -> tasks --------
t0 = time.time()
plan.show()                                 # show() is an ACTION: triggers compute
print("action ran the job in %.3fs" % (time.time() - t0))
# In the Spark UI (printed URL, /jobs tab) this job shows:
#   Stage 1 = 8 map tasks   (one per input partition, narrow filter+partial agg)
#   --- shuffle / Exchange ---
#   Stage 2 = up to 200 reduce tasks (one per shuffle partition; only 5 non-empty)

# --- PITFALL: a second action re-runs the WHOLE plan from scratch --------
plan.count()        # re-reads + re-shuffles everything again, no reuse!
# Fix: cache the shared result once, materialize it, then reuse.
cached = plan.cache()
cached.count()      # first action populates the cache
cached.show()       # this one reads from memory, not a full re-compute

print(spark.sparkContext.uiWebUrl)          # open the Spark UI here
spark.stop()

## Visualize the data & results

_When an action fires a filter+groupBy query, how does Spark break the JOB into STAGES (split at the shuffle) and TASKS (one per partition), and how much data actually moves?_

In [ ]:
import numpy as np

# --- Job shape: a 12M-row table in 8 input partitions -------------------
N        = 12_000_000     # total rows
P_in     = 8              # input partitions  -> Stage 1 has one map task each
shuffle_p = 200           # spark.sql.shuffle.partitions (Spark default)
regions  = 5              # distinct group keys -> only this many non-empty reducers
pass_rate = 0.70          # fraction of rows surviving filter(amount > 0)

# --- Stage / task counts (stages split at the shuffle boundary) ----------
stage1_tasks   = P_in                 # NARROW filter: one task per input partition
stage2_tasks   = shuffle_p            # one reduce task per shuffle partition
nonempty_reduce = regions             # only the keys that exist do real work
print("Stage 1 map tasks   :", stage1_tasks)    # -> 8
print("Stage 2 reduce tasks :", stage2_tasks)   # -> 200
print("  ... non-empty      :", nonempty_reduce) # -> 5

# --- Data moved: narrow shuffles nothing, wide shuffles the survivors ----
rows_after_filter = int(N * pass_rate)           # -> 8,400,000 rows survive
bytes_per_row     = 24                            # region key + partial sum, approx
narrow_moved_mb   = 0.0                           # filter is per-partition: 0 network
wide_moved_mb     = rows_after_filter * bytes_per_row / 1e6
print("rows after filter   :", rows_after_filter)        # -> 8,400,000
print("narrow moved (MB)   :", round(narrow_moved_mb, 1)) # -> 0.0
print("wide   moved (MB)   :", round(wide_moved_mb, 1))   # -> 201.6
# Stage/task bars: [8, 200, 5]   Data-moved bars: [0.0, 201.6]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You run df2 = df.filter("country = 'US'").select("user_id", "amount") and it returns in a few milliseconds with no error. A teammate says "great, the filter ran." Are they right? What actually happened, and when will it run?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify whether filter and select are transformations or actions. — _Both are transformations &mdash; they are lazy and compute nothing on their own._
- Recognize that returning instantly with no error means no data was processed. — _Spark only appended two steps to the DAG (Directed Acyclic Graph); it did not read or filter any rows._
- Note that the work fires only on the first action (e.g. df2.show() or df2.count()). — _Actions trigger Spark to plan and run the whole chain; that is also when a bad column name would finally error._

**Answer:** No. filter and select are transformations, so they are lazy &mdash; they only added two steps to the DAG and computed nothing, which is why it returned in milliseconds. The actual reading and filtering happen later, on the first action such as df2.show(), df2.count(), or df2.write(...). A latent error (say a misspelled column) would also only surface then, not now.

</details>

**Problem 2.** In a loop you do for c in countries: print(df.filter(df.country==c).count()) on the same large uncached df, and it is painfully slow. Why, and what one change fixes most of it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that count() is an action and the loop calls it once per country. — _Each action re-executes the entire plan from the original source &mdash; the read is repeated every iteration._
- Realize df is uncached, so nothing from the previous pass is reused. — _Without caching, Spark re-reads and re-computes the base DataFrame on every single action._
- Cache the shared frame once before the loop: df.cache(); df.count() to materialize it, then loop. — _The first action populates the cache; later filters read from memory instead of re-scanning the source._

**Answer:** Each count() is an action, and every action re-runs the whole plan from the source &mdash; so an uncached df is fully re-read on every loop iteration. The fix is to cache the shared frame once: df.cache() then a single df.count() to materialize it, after which the per-country filters read from memory. (Better still, do it in one pass: df.groupBy("country").count(), a single action.)

</details>

**Problem 3.** Reading a query plan from .explain() for df.filter(...).groupBy("region").agg(...), you see one Exchange hashpartitioning(region, 200) node. How many stages does this job have, and which transformation is narrow vs wide?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that Spark splits a job into stages at every shuffle boundary, and a shuffle appears as an Exchange. — _One Exchange means exactly one shuffle, which cuts the job into two stages._
- Classify filter: each output partition comes from one input partition, no data moves. — _That is the definition of a narrow transformation &mdash; it lives before the Exchange, in Stage 1._
- Classify groupBy: rows for the same region must be brought together across partitions. — _That requires moving data across the cluster &mdash; a wide transformation, which is the Exchange itself._

**Answer:** Two stages. The single Exchange is the one shuffle, and stages split at shuffle boundaries: Stage 1 is everything before it, Stage 2 everything after. The filter is narrow (each output partition comes from one input partition, no data movement) and lives in Stage 1; the groupBy is wide (it shuffles rows so each region lands together) and is exactly the Exchange that begins Stage 2.

</details>